In [1]:
!pip install pandas tqdm requests -q

In [2]:
import os

DATA_PATH = "TACO/data"

print(os.listdir(DATA_PATH)[:20])

['all_image_urls.csv', 'annotations.json', 'annotations_unofficial.json', '.ipynb_checkpoints', 'batch.zip', 'batch_3', 'batch_2', 'batch_1', 'batch_4', 'batch_6', 'batch_7', 'batch_5', 'batch_8', 'batch_9', 'batch_10', 'batch_11', 'batch_12', 'batch_13', 'batch_14', 'batch_15']


In [3]:
import os

print(os.getcwd())
print(os.listdir("TACO/data"))

/home/sagemaker-user/Classification-and-Sorting-of-Recyclables-from-Trash-
['all_image_urls.csv', 'annotations.json', 'annotations_unofficial.json', '.ipynb_checkpoints', 'batch.zip', 'batch_3', 'batch_2', 'batch_1', 'batch_4', 'batch_6', 'batch_7', 'batch_5', 'batch_8', 'batch_9', 'batch_10', 'batch_11', 'batch_12', 'batch_13', 'batch_14', 'batch_15']


In [4]:
import os

TACO_PATH = "TACO/data"
ANNOTATION_FILE = os.path.join(TACO_PATH, "annotations.json")
IMAGE_DIR = TACO_PATH
OUTPUT_DATASET = "dataset"

In [5]:
import json
import os

print("Annotation exists:", os.path.exists(ANNOTATION_FILE))
print("TACO path exists:", os.path.exists(TACO_PATH))
print("Top items:", os.listdir(TACO_PATH)[:10])

with open(ANNOTATION_FILE, "r") as f:
    data = json.load(f)

print("Images:", len(data["images"]))
print("Annotations:", len(data["annotations"]))
print("Categories:", len(data["categories"]))

print("Sample image entry:")
print(data["images"][0])

Annotation exists: True
TACO path exists: True
Top items: ['all_image_urls.csv', 'annotations.json', 'annotations_unofficial.json', '.ipynb_checkpoints', 'batch.zip', 'batch_3', 'batch_2', 'batch_1', 'batch_4', 'batch_6']
Images: 1500
Annotations: 4784
Categories: 60
Sample image entry:
{'id': 0, 'width': 1537, 'height': 2049, 'file_name': 'batch_1/000006.jpg', 'license': None, 'flickr_url': 'https://farm66.staticflickr.com/65535/33978196618_e30a59e0a8_o.png', 'coco_url': None, 'date_captured': None, 'flickr_640_url': 'https://farm66.staticflickr.com/65535/33978196618_632623b4fc_z.jpg'}


In [6]:
def find_image_path(file_name):
    path = os.path.join(IMAGE_DIR, file_name)
    if os.path.exists(path):
        return path
    return None

In [7]:
sample_file = data["images"][0]["file_name"]
print("Sample file:", sample_file)
print("Found path:", find_image_path(sample_file))

Sample file: batch_1/000006.jpg
Found path: TACO/data/batch_1/000006.jpg


In [8]:
CLASS_NAMES = [
    "paper_cardboard",
    "plastic_items",
    "glass_metal",
    "general_waste"
]

In [9]:
images = {img["id"]: img for img in data["images"]}
categories = {cat["id"]: cat["name"] for cat in data["categories"]}

print("First 20 categories:")
for i, (cid, cname) in enumerate(categories.items()):
    if i == 20:
        break
    print(cid, cname)

First 20 categories:
0 Aluminium foil
1 Battery
2 Aluminium blister pack
3 Carded blister pack
4 Other plastic bottle
5 Clear plastic bottle
6 Glass bottle
7 Plastic bottle cap
8 Metal bottle cap
9 Broken glass
10 Food Can
11 Aerosol
12 Drink can
13 Toilet tube
14 Other carton
15 Egg carton
16 Drink carton
17 Corrugated carton
18 Meal carton
19 Pizza box


In [32]:
CATEGORY_MAPPING = {}

for name in categories.values():
    n = name.lower()

    if "paper" in n or "cardboard" in n or "carton" in n:
        CATEGORY_MAPPING[name] = "paper_cardboard"
    elif "plastic" in n or "bag" in n:
        CATEGORY_MAPPING[name] = "plastic_items"
    elif "glass" in n or "metal" in n or "can" in n:
        CATEGORY_MAPPING[name] = "glass_metal"
    else:
        CATEGORY_MAPPING[name] = "general_waste"

print(CATEGORY_MAPPING)

{'Aluminium foil': 'general_waste', 'Battery': 'general_waste', 'Aluminium blister pack': 'general_waste', 'Carded blister pack': 'general_waste', 'Other plastic bottle': 'plastic_items', 'Clear plastic bottle': 'plastic_items', 'Glass bottle': 'glass_metal', 'Plastic bottle cap': 'plastic_items', 'Metal bottle cap': 'glass_metal', 'Broken glass': 'glass_metal', 'Food Can': 'glass_metal', 'Aerosol': 'general_waste', 'Drink can': 'glass_metal', 'Toilet tube': 'general_waste', 'Other carton': 'paper_cardboard', 'Egg carton': 'paper_cardboard', 'Drink carton': 'paper_cardboard', 'Corrugated carton': 'paper_cardboard', 'Meal carton': 'paper_cardboard', 'Pizza box': 'general_waste', 'Paper cup': 'paper_cardboard', 'Disposable plastic cup': 'plastic_items', 'Foam cup': 'general_waste', 'Glass cup': 'glass_metal', 'Other plastic cup': 'plastic_items', 'Food waste': 'general_waste', 'Glass jar': 'glass_metal', 'Plastic lid': 'plastic_items', 'Metal lid': 'glass_metal', 'Other plastic': 'plasti

In [33]:
from pathlib import Path

for split in ["train", "val", "test"]:
    for cls in CLASS_NAMES:
        Path(f"{OUTPUT_DATASET}/{split}/{cls}").mkdir(parents=True, exist_ok=True)

print("Folders created.")

Folders created.


In [34]:
import random

image_ids = list(images.keys())
random.seed(42)
random.shuffle(image_ids)

train_split = int(0.7 * len(image_ids))
val_split = int(0.85 * len(image_ids))

train_ids = image_ids[:train_split]
val_ids = image_ids[train_split:val_split]
test_ids = image_ids[val_split:]

print(len(train_ids), len(val_ids), len(test_ids))

1050 225 225


In [35]:
from collections import defaultdict

ann_by_image = defaultdict(list)

for ann in data["annotations"]:
    cat_name = categories[ann["category_id"]]
    mapped = CATEGORY_MAPPING.get(cat_name, "general_waste")
    ann_by_image[ann["image_id"]].append((mapped, ann["bbox"]))

print("Images with annotations:", len(ann_by_image))

Images with annotations: 1500


In [36]:
from PIL import Image
from tqdm import tqdm

def process_split(ids, split):
    saved = 0

    for img_id in tqdm(ids, desc=f"Processing {split}"):
        if img_id not in ann_by_image:
            continue

        img_info = images[img_id]
        img_path = find_image_path(img_info["file_name"])

        if img_path is None:
            continue

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        for i, (cls, bbox) in enumerate(ann_by_image[img_id]):
            x, y, w, h = map(int, bbox)

            if w <= 5 or h <= 5:
                continue

            crop = img.crop((x, y, x + w, y + h))
            save_path = f"{OUTPUT_DATASET}/{split}/{cls}/{img_id}_{i}.jpg"

            try:
                crop.save(save_path)
                saved += 1
            except Exception:
                pass

    print(f"{split}: saved {saved} crops")

process_split(train_ids, "train")
process_split(val_ids, "val")
process_split(test_ids, "test")

Processing train: 100%|██████████| 1050/1050 [01:26<00:00, 12.17it/s]


train: saved 3215 crops


Processing val: 100%|██████████| 225/225 [00:19<00:00, 11.61it/s]


val: saved 822 crops


Processing test: 100%|██████████| 225/225 [00:17<00:00, 12.52it/s]

test: saved 746 crops


In [37]:
for split in ["train", "val", "test"]:
    print(f"\n{split}")
    for cls in CLASS_NAMES:
        folder = os.path.join(OUTPUT_DATASET, split, cls)
        print(cls, len(os.listdir(folder)))


train
paper_cardboard 321
plastic_items 1336
glass_metal 482
general_waste 1076

val
paper_cardboard 51
plastic_items 369
glass_metal 75
general_waste 327

test
paper_cardboard 75
plastic_items 305
glass_metal 70
general_waste 296


In [38]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder(f"{OUTPUT_DATASET}/train", transform=train_transform)
val_ds = datasets.ImageFolder(f"{OUTPUT_DATASET}/val", transform=test_transform)
test_ds = datasets.ImageFolder(f"{OUTPUT_DATASET}/test", transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(train_ds.classes)
print(len(train_ds), len(val_ds), len(test_ds))

['general_waste', 'glass_metal', 'paper_cardboard', 'plastic_items']
3215 822 746


In [40]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

cpu


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/sagemaker-user/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]


In [1]:
for epoch in range(5):
    model.train()
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    print(f"Epoch {epoch+1}: Train Acc = {correct/total:.4f}")

NameError: name 'model' is not defined

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total

train_acc = evaluate(model, train_loader)
val_acc = evaluate(model, val_loader)
test_acc = evaluate(model, test_loader)

print("Train:", train_acc)
print("Val:", val_acc)
print("Test:", test_acc)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "CNN Architecture": ["ResNet18"],
    "Train Accuracy": [train_acc],
    "Validation Accuracy": [val_acc],
    "Test Accuracy": [test_acc]
})

results